# LeetCode 3954 — Sum of Compatible Numbers in Range I bằng Python

Link đề: https://leetcode.com/problems/sum-of-compatible-numbers-in-range-i/ · Độ khó: **Easy** · Chủ đề: Math, Bit Manipulation, Enumeration

Skeleton LeetCode:

```python
class Solution:
    def sumOfGoodIntegers(self, n: int, k: int) -> int:
        ...
```

> Ghi chú: tên bài, ví dụ, constraints và chữ ký hàm được đối chiếu qua leetcode.doocs.org (Easy, không Premium); phần phát biểu đề được viết lại bằng lời của mình. Slug URL được suy ra từ tên bài.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: số nguyên dương `x` là **tương thích** với `n` nếu `|x - n| <= k` **và** `n & x == 0`; trả về tổng mọi số tương thích.
2. Biết toán tử AND bit `&`, ý nghĩa của `n & x == 0` (hai số **không chung bit 1 nào**) và dùng `range` với cận đúng (`max(1, n - k)` đến `n + k` **bao gồm**).
3. Hiểu mẹo **liệt kê tập con của mặt nạ** (`sub = (sub - 1) & mask`) để chỉ duyệt các `x` có thể thỏa `n & x == 0`.
4. Viết 4 cách (duyệt theo khoảng, duyệt rộng rồi lọc, tập con của mặt nạ, tổng bằng `sum` + comprehension) và đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh lỗi: để khoảng bắt đầu âm hoặc `0`, quên cận trên `n + k` (bao gồm), nhầm `&` với `^`, nhầm `|x - n|` thành `x - n`.

## 2. Đề bài & ví dụ

Cho hai số nguyên `n` và `k`. Một số nguyên **dương** `x` gọi là **tương thích** với `n` nếu thỏa **cả hai** điều kiện:

1. khoảng cách `|x - n| <= k`;
2. `n & x == 0` (phép AND bit của `n` và `x` bằng `0`: không có vị trí bit nào mà cả hai cùng là `1`).

Trả về **tổng** của mọi số nguyên `x` tương thích.

| Ví dụ | `n`, `k` | Output | Giải thích |
|---|---|---|---|
| 1 | `n=2, k=3` | `10` | `x` nằm trong `[1, 5]`. `1 (001)`, `4 (100)`, `5 (101)` đều AND với `2 (010)` bằng `0`; `2, 3` thì không. Tổng `1 + 4 + 5 = 10` |
| 2 | `n=5, k=1` | `0` | `x` trong `[4, 6]`; `4 & 5 = 4`, `5 & 5 = 5`, `6 & 5 = 4` đều khác `0` nên tổng `0` |

Constraints: `1 <= n <= 100`; `1 <= k <= 100`.

Lưu ý: `x = n` không bao giờ tương thích vì `n & n = n > 0`.

In [1]:
n, k = 2, 3
for x in range(max(1, n - k), n + k + 1):
    print(f"x={x}: {x:03b} & {n:03b} = {x & n}  ->  {'tương thích' if x & n == 0 else 'loại'}")

x=1: 001 & 010 = 0  ->  tương thích
x=2: 010 & 010 = 2  ->  loại
x=3: 011 & 010 = 2  ->  loại
x=4: 100 & 010 = 0  ->  tương thích
x=5: 101 & 010 = 0  ->  tương thích


## 3. Kiến thức nền

### 3.1. AND bit `&` và điều kiện `n & x == 0`

`a & b` giữ lại bit `1` chỉ ở những vị trí **cả hai** là `1`. `n & x == 0` nghĩa là **tập vị trí bit 1 của `x` rời nhau với của `n`**: `x` chỉ được bật bit ở những vị trí mà `n` là `0`.

In [2]:
n = 5                                            # 101
for x in (2, 8, 3, 10):
    print(f"x={x:>2} ({x:04b}) & n ({n:04b}) = {x & n:04b}  -> {x & n == 0}")

x= 2 (0010) & n (0101) = 0000  -> True
x= 8 (1000) & n (0101) = 0000  -> True
x= 3 (0011) & n (0101) = 0001  -> False
x=10 (1010) & n (0101) = 0000  -> True


### 3.2. `range(a, b)` **không** bao gồm `b`

`range(max(1, n - k), n + k + 1)` đi từ `max(1, n - k)` đến `n + k` (cận trên cộng thêm 1 vì `range` bỏ phần tử cuối). Số `x` phải **dương** nên cận dưới là `max(1, n - k)`: nếu `n - k` là `0` hoặc âm thì bắt đầu từ `1`.

In [3]:
print(list(range(2, 6)), list(range(max(1, 2 - 3), 2 + 3 + 1)))

[2, 3, 4, 5] [1, 2, 3, 4, 5]


### 3.3. Số âm và phép AND

Trong Python, số nguyên âm có biểu diễn bù hai "vô hạn bit 1 ở bên trái". Ví dụ `-4 & 2 == 0` (đúng!) dù `-4` không dương. Nếu quên giới hạn `x >= 1`, các số âm có thể lọt vào tổng và làm sai kết quả.

In [4]:
print(-4 & 2, -3 & 2, -1 & 2, 0 & 2)

0 0 2 0


### 3.4. Mặt nạ (mask), phần bù và `bit_length`

`n.bit_length()` là số bit cần để viết `n`. `(1 << L) - 1` là mặt nạ gồm `L` bit `1`. Tập các vị trí bit mà `n` là `0` (trong `L` bit) được cho bởi `mask = ((1 << L) - 1) & ~n`. Mọi `x` thỏa `n & x == 0` và `x < 2^L` đều là **tập con** của `mask`.

In [5]:
n = 2
L = (n + 3).bit_length()
full = (1 << L) - 1
mask = full & ~n
print(L, bin(full), bin(mask))

3 0b111 0b101


### 3.5. Liệt kê mọi tập con của mặt nạ

Vòng lặp `sub = mask; while True: ...; if sub == 0: break; sub = (sub - 1) & mask` đi qua **mọi** tập con của `mask` (từ lớn xuống nhỏ, kể cả `0`). Số tập con là `2^{số bit 1 của mask}`.

In [6]:
mask = 0b101
sub = mask
subs = []
while True:
    subs.append(sub)
    if sub == 0:
        break
    sub = (sub - 1) & mask
print([bin(s) for s in subs])

['0b101', '0b100', '0b1', '0b0']


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Miền đáng xét rất nhỏ: `x` chỉ nằm trong `[n - k, n + k]`, tối đa `2k + 1 <= 201` số. Cách tự nhiên nhất: **thử từng `x`** trong khoảng đó, giữ `x` nếu `n & x == 0`, rồi cộng.

Tính tay `n = 2, k = 3`: khoảng `[-1, 5]` nhưng chỉ lấy `x >= 1` nên `[1, 5]`.

| `x` | nhị phân | `x & 2` | giữ? |
|---|---|---|---|
| 1 | `001` | 0 | có |
| 2 | `010` | 2 | không |
| 3 | `011` | 2 | không |
| 4 | `100` | 0 | có |
| 5 | `101` | 0 | có |

Tổng `1 + 4 + 5 = 10`. Tính tay `n = 5, k = 1`: `x` trong `[4, 6]`, ba số đều chung bit với `101` -> `0`.

In [7]:
def explain(n, k):
    kept = [x for x in range(max(1, n - k), n + k + 1) if x & n == 0]
    print(f"n={n}, k={k}: giữ {kept}, tổng = {sum(kept)}")

explain(2, 3)
explain(5, 1)
explain(8, 4)

n=2, k=3: giữ [1, 4, 5], tổng = 10
n=5, k=1: giữ [], tổng = 0
n=8, k=4: giữ [4, 5, 6, 7], tổng = 22


## 5. Cách 1 — Brute Force / baseline: duyệt đúng khoảng `[max(1, n-k), n+k]`

1. `ans = 0`.
2. Với mỗi `x` từ `max(1, n - k)` đến `n + k` (bao gồm hai đầu), nếu `n & x == 0` thì `ans += x`.
3. Trả `ans`.

Đây gần như dịch nguyên văn đề, `O(k)` thời gian. Với `k <= 100` là quá đủ nhanh; "nút thắt" gần như không có, nhưng ta sẽ thử vài cách khác để hiểu cấu trúc bit.

In [8]:
class SolutionRange:
    def sumOfGoodIntegers(self, n, k):
        ans = 0
        for x in range(max(1, n - k), n + k + 1):
            if n & x == 0:
                ans += x
        return ans

print(SolutionRange().sumOfGoodIntegers(2, 3), SolutionRange().sumOfGoodIntegers(5, 1))

10 0


**Độ phức tạp:** Time `O(k)` (tối đa `2k + 1` số), Space `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Quan sát 1 (miền):** mọi `x` hợp lệ nằm trong khoảng `[max(1, n - k), n + k]` — chỉ cần xét đúng khoảng đó, không cần duyệt từ `1` đến vô cùng.

**Quan sát 2 (cấu trúc bit):** `n & x == 0` nghĩa là `x` chỉ được bật các bit mà `n` đang **tắt**. Do `x <= n + k`, độ dài bit của `x` không vượt `L = (n + k).bit_length()`. Gọi `mask = ((1 << L) - 1) & ~n`. Khi đó **tập mọi `x` thỏa `n & x == 0` và `x <= 2^L - 1` chính là tập mọi tập con của `mask`**. Ta có thể duyệt các tập con rồi chỉ giữ những `x` trong khoảng `[n - k, n + k]` và `x >= 1`.

**Quy tắc "nếu... thì...":** nếu `x` là tập con của `mask` và `x` thuộc `[max(1, n - k), n + k]` thì cộng `x`.

**Invariant:** mỗi `x` được xét đúng một lần (mỗi tập con xuất hiện một lần), và `x = 0` bị loại bởi điều kiện dương.

## 7. Các cách tiếp theo

### Cách 2 — Duyệt rộng rồi lọc bằng điều kiện đầy đủ

Duyệt `x` từ `1` đến `n + k`, kiểm tra **cả hai** điều kiện `abs(x - n) <= k` và `n & x == 0`. Dễ đúng, ít phải nghĩ tới cận dưới, nhưng duyệt nhiều hơn.

In [9]:
class SolutionFilter:
    def sumOfGoodIntegers(self, n, k):
        return sum(x for x in range(1, n + k + 1) if abs(x - n) <= k and n & x == 0)

print(SolutionFilter().sumOfGoodIntegers(2, 3), SolutionFilter().sumOfGoodIntegers(5, 1))

10 0


**Độ phức tạp:** Time `O(n + k)`, Space `O(1)`.

### Cách 3 — Liệt kê tập con của mặt nạ bit

Dựng `mask` rồi duyệt các tập con `x` (kể cả `0`); giữ `x` nếu `x >= 1` và `|x - n| <= k`. Số tập con là `2^{số bit 0 của n trong L bit}`, với `n + k <= 200` nên `L <= 8`, tối đa 256 tập con.

In [10]:
class SolutionSubmask:
    def sumOfGoodIntegers(self, n, k):
        L = (n + k).bit_length()
        mask = ((1 << L) - 1) & ~n
        ans = 0
        sub = mask
        while True:
            if sub >= 1 and abs(sub - n) <= k:
                ans += sub
            if sub == 0:
                break
            sub = (sub - 1) & mask
        return ans

print(SolutionSubmask().sumOfGoodIntegers(2, 3), SolutionSubmask().sumOfGoodIntegers(5, 1))

10 0


**Độ phức tạp:** Time `O(2^b)` với `b` = số bit 0 của `n` trong `L` bit (`b <= 8`), Space `O(1)`. Cách này hữu ích khi `k` rất lớn mà `n` có nhiều bit 1, nhưng với đề này không nhanh hơn Cách 1 nhiều.

### Cách 4 — `sum` + list comprehension, không biến tạm

Cách gọn của Cách 1: `sum(x for x in range(max(1, n - k), n + k + 1) if not n & x)`.

In [11]:
class SolutionOneLine:
    def sumOfGoodIntegers(self, n, k):
        return sum(x for x in range(max(1, n - k), n + k + 1) if not (n & x))

print(SolutionOneLine().sumOfGoodIntegers(2, 3), SolutionOneLine().sumOfGoodIntegers(5, 1))

10 0


**Độ phức tạp:** Time `O(k)`, Space `O(1)` (generator).

## 8. TRACE từng bước

Trace Cách 1 với `n = 2, k = 3`, khoảng `x = 1..5`:

| `x` | `x & n` | cộng? | `ans` |
|---|---|---|---|
| 1 | 0 | có | 1 |
| 2 | 2 | không | 1 |
| 3 | 2 | không | 1 |
| 4 | 0 | có | 5 |
| 5 | 0 | có | 10 |

In [12]:
def trace_range(n, k):
    lo, hi = max(1, n - k), n + k
    print(f"n={n} ({n:b}), k={k}, khoảng x = {lo}..{hi}")
    ans = 0
    for x in range(lo, hi + 1):
        a = n & x
        take = a == 0
        if take:
            ans += x
        print(f"  x={x:>3} ({x:b}): n&x={a} -> {'cộng' if take else 'bỏ  '} ans={ans}")
    return ans

assert trace_range(2, 3) == 10
assert trace_range(5, 1) == 0

n=2 (10), k=3, khoảng x = 1..5
  x=  1 (1): n&x=0 -> cộng ans=1
  x=  2 (10): n&x=2 -> bỏ   ans=1
  x=  3 (11): n&x=2 -> bỏ   ans=1
  x=  4 (100): n&x=0 -> cộng ans=5
  x=  5 (101): n&x=0 -> cộng ans=10
n=5 (101), k=1, khoảng x = 4..6
  x=  4 (100): n&x=4 -> bỏ   ans=0
  x=  5 (101): n&x=5 -> bỏ   ans=0
  x=  6 (110): n&x=4 -> bỏ   ans=0


In [13]:
def trace_submask(n, k):
    L = (n + k).bit_length()
    mask = ((1 << L) - 1) & ~n
    print(f"n={n:b}, L={L}, mask={mask:b}")
    ans, sub = 0, mask
    while True:
        ok = sub >= 1 and abs(sub - n) <= k
        print(f"  sub={sub:>3} ({sub:b}), |sub-n|={abs(sub - n)}, giữ? {ok}")
        if ok:
            ans += sub
        if sub == 0:
            break
        sub = (sub - 1) & mask
    print("  tổng =", ans)
    return ans

assert trace_submask(2, 3) == 10

n=10, L=3, mask=101
  sub=  5 (101), |sub-n|=3, giữ? True
  sub=  4 (100), |sub-n|=2, giữ? True
  sub=  1 (1), |sub-n|=1, giữ? True
  sub=  0 (0), |sub-n|=2, giữ? False
  tổng = 10


## 9. Code nộp LeetCode

Dùng Cách 1 (rõ ràng, `O(k)`). Lưu ý tên phương thức trên LeetCode là `sumOfGoodIntegers` theo mẫu chữ ký đã đối chiếu.

In [14]:
from typing import List, Optional


class Solution:
    def sumOfGoodIntegers(self, n: int, k: int) -> int:
        ans = 0
        for x in range(max(1, n - k), n + k + 1):
            if n & x == 0:
                ans += x
        return ans


print(Solution().sumOfGoodIntegers(2, 3), Solution().sumOfGoodIntegers(5, 1))

10 0


## 10. Vì sao thuật toán đúng

1. **Miền đủ.** Điều kiện `|x - n| <= k` tương đương `n - k <= x <= n + k`. Cộng với `x >= 1` (dương), tập ứng viên là `[max(1, n - k), n + k]`.
2. **Không thừa, không thiếu.** `range(lo, hi + 1)` đi qua mọi số nguyên trong `[lo, hi]` đúng một lần (cận `hi + 1` vì `range` loại cận trên).
3. **Điều kiện bit.** Với mỗi ứng viên, kiểm tra `n & x == 0` đúng định nghĩa; chỉ cộng những `x` thỏa.
4. **Cách 3 (tập con) tương đương.** Mọi `x` thỏa `n & x == 0` với `0 <= x < 2^L` là tập con của `mask = ~n` giới hạn trong `L` bit, và ngược lại mọi tập con của `mask` thỏa điều kiện. Vì `x <= n + k < 2^L` nên không bỏ sót `x` nào trong khoảng.
5. **Không đếm trùng.** Mỗi `x` xuất hiện một lần trong mỗi cách duyệt.

## 11. Phân tích Time / Space Complexity

- Cách 1 (nộp): Time `O(min(k, n) + k) = O(k)` (không quá `2k + 1 <= 201` phép thử), Space `O(1)`.
- Cách 2: Time `O(n + k)`, Space `O(1)`.
- Cách 3: Time `O(2^b)` với `b <= 8`, Space `O(1)`.
- Cách 4: Time `O(k)`, Space `O(1)`.

Với `n, k <= 100` mọi cách đều dưới 300 bước. "Phiên bản II" của bài (nếu có, với `n, k` rất lớn) sẽ đòi hỏi cách tiếp cận bit/toán học thay vì duyệt khoảng; Cách 3 là bước làm quen với ý tưởng đó.

## 12. Bộ test

### Cách kiểm thử trong notebook này

Chúng ta dùng ba lớp kiểm thử, từ dễ đến khó:

1. **Test cố định**: ví dụ của đề + các trường hợp biên tự nghĩ ra. Mỗi test là một `assert hàm(input) == kết_quả_mong_đợi`. Nếu sai, Python ném `AssertionError` và dừng ngay, kèm thông tin ta in ra để dễ lần lại.
2. **Test ngẫu nhiên có `random.seed`**: gọi `random.seed(số)` trước vòng lặp để **mỗi lần chạy lại sinh đúng cùng dữ liệu** (tái lập được). Input luôn được sinh **trong đúng constraints** của đề.
3. **Đối chiếu nhiều cách (oracle)**: một cách rất đơn giản, hiển nhiên đúng (brute force) đóng vai "trọng tài"; mọi cách nhanh hơn phải cho cùng kết quả với trọng tài trên mọi input ngẫu nhiên.

In [15]:
import random

def ref(n, k):
    # trọng tài: duyệt rộng thẳng theo định nghĩa trên miền x dương lớn hơn cần thiết
    total = 0
    for x in range(1, 2 * (n + k) + 5):
        if abs(x - n) <= k and (n & x) == 0:
            total += x
    return total

solutions = [SolutionRange(), SolutionFilter(), SolutionSubmask(), SolutionOneLine(), Solution()]

fixed = [
    ((2, 3), 10), ((5, 1), 0), ((1, 1), 2), ((1, 2), 2), ((1, 100), None), ((100, 100), None),
    ((7, 100), None), ((8, 4), None), ((3, 1), 4), ((4, 4), None), ((100, 1), None),
]
for (n, k), expected in fixed:
    if expected is None:
        expected = ref(n, k)
    for sol in solutions:
        assert sol.sumOfGoodIntegers(n, k) == expected, (type(sol).__name__, n, k)
print("test cố định OK:", len(fixed), "ca")

for n in range(1, 101):                         # vét cạn một phần miền
    for k in (1, 2, 5, 17, 50, 100):
        assert Solution().sumOfGoodIntegers(n, k) == ref(n, k), (n, k)
print("vét cạn n=1..100, k in {1,2,5,17,50,100} OK")

random.seed(3954)
for _ in range(500):
    n, k = random.randint(1, 100), random.randint(1, 100)
    expected = ref(n, k)
    for sol in solutions:
        assert sol.sumOfGoodIntegers(n, k) == expected, (type(sol).__name__, n, k)
print("test ngẫu nhiên OK")

test cố định OK: 11 ca
vét cạn n=1..100, k in {1,2,5,17,50,100} OK
test ngẫu nhiên OK


### Test thuộc tính (property test)

(1) Đáp án luôn `>= 0`. (2) Khi `k` tăng, đáp án **không giảm** (thêm ứng viên chỉ làm tăng tổng vì mọi `x` dương). (3) Với `n` là lũy thừa của 2, `n = 2^p`, mọi `x < n` đều tương thích nếu nằm trong khoảng (vì `x` không có bit `p`). (4) `x = n` không bao giờ góp vào tổng.

In [16]:
random.seed(39540)
for _ in range(400):
    n = random.randint(1, 100)
    ks = sorted(random.randint(1, 100) for _ in range(4))
    vals = [Solution().sumOfGoodIntegers(n, k) for k in ks]
    assert all(v >= 0 for v in vals) and vals == sorted(vals)
for p in range(0, 7):
    n = 1 << p
    for k in (1, 5, 40):
        expected = sum(x for x in range(max(1, n - k), n) if True) + sum(x for x in range(n + 1, n + k + 1) if (n & x) == 0)
        assert Solution().sumOfGoodIntegers(n, k) == expected, (n, k)
print("property test OK")

property test OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: quên cận trên `n + k` (dùng `range(..., n + k)`)

`range` bỏ phần tử cuối, nên `x = n + k` bị bỏ sót. Với `n = 2, k = 3`, bỏ `x = 5` làm kết quả `5` thay vì `10`.

In [17]:
def wrong_exclusive(n, k):
    return sum(x for x in range(max(1, n - k), n + k) if n & x == 0)       # SAI: thiếu n + k

print(wrong_exclusive(2, 3), "!=", Solution().sumOfGoodIntegers(2, 3))
assert wrong_exclusive(2, 3) == 5 and Solution().sumOfGoodIntegers(2, 3) == 10

5 != 10


### Lỗi 2: để khoảng bắt đầu ở `n - k` (có thể `<= 0`)

Đề yêu cầu `x` **dương**. Khi `n - k <= 0`, `range(n - k, ...)` chứa `0` và số âm. Vì `-3 & 2 == 0` (xem mục 3.3), một số âm lọt vào và **trừ** vào tổng.

In [18]:
def wrong_negative(n, k):
    return sum(x for x in range(n - k, n + k + 1) if n & x == 0)            # SAI: không giới hạn x >= 1

print(wrong_negative(2, 6), "!=", Solution().sumOfGoodIntegers(2, 6))
assert wrong_negative(2, 6) != Solution().sumOfGoodIntegers(2, 6)
assert Solution().sumOfGoodIntegers(2, 6) == 1 + 4 + 5 + 8

11 != 18


### Lỗi 3: nhầm `&` với `^` hoặc `|`

`n ^ x == 0` chỉ đúng khi `x == n`; `n | x == 0` chỉ đúng khi cả hai bằng 0. Điều kiện đúng là **AND bằng 0**.

In [19]:
def wrong_xor(n, k):
    return sum(x for x in range(max(1, n - k), n + k + 1) if n ^ x == 0)    # SAI: XOR bằng 0 nghĩa là x == n

print(wrong_xor(2, 3), "!=", Solution().sumOfGoodIntegers(2, 3))
assert wrong_xor(2, 3) == 2 and Solution().sumOfGoodIntegers(2, 3) == 10

2 != 10


### Lỗi 4: quên `abs` hoặc chỉ xét một phía của khoảng

Chỉ xét `x` từ `n` trở lên (`x - n <= k`) bỏ sót các `x < n` như `x = 1` ở ví dụ 1.

In [20]:
def wrong_one_side(n, k):
    return sum(x for x in range(n, n + k + 1) if n & x == 0)                # SAI: bỏ các x nhỏ hơn n

print(wrong_one_side(2, 3), "!=", Solution().sumOfGoodIntegers(2, 3))
assert wrong_one_side(2, 3) == 9 and Solution().sumOfGoodIntegers(2, 3) == 10

9 != 10


### Lỗi 5: nhầm độ ưu tiên toán tử (kiểu C/C++)

Trong C/C++, `n & x == 0` được hiểu là `n & (x == 0)`. Trong **Python**, `&` ưu tiên cao hơn `==` nên `n & x == 0` đúng nghĩa. Nhưng khi chuyển bài sang C++/Java phải viết `(n & x) == 0`. Thói quen tốt: **luôn đặt ngoặc**.

In [21]:
n, x = 2, 4
print(n & x == 0, (n & x) == 0, n & (x == 0))
assert (n & x == 0) == ((n & x) == 0)

True True 0


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Ràng buộc nhỏ (`n, k <= 100`) + điều kiện đơn giản trên từng số -> **vét cạn có cận chuẩn**: viết cho đúng cận (`max(1, ...)`, `+ 1`) trước khi tối ưu.
- Điều kiện `a & b == 0` -> "hai tập bit rời nhau" -> `b` là tập con của `~a` (trong số bit cần thiết).
- Duyệt mọi tập con của mask: `sub = (sub - 1) & mask` (mẫu kinh điển trong bài toán mặt nạ).
- Với đề có chữ "dương", luôn nhớ `x >= 1` và thử một ca có `n - k <= 0`.
- `range(a, b)` loại `b`: viết `b + 1` khi muốn gồm `b`.
- Tự viết **trọng tài** đơn giản (duyệt rộng) để so sánh cách tối ưu: kỹ thuật dùng suốt notebook này.

## 15. Biến thể của bài

1. Phiên bản **"Range II"** (nếu có, với `n, k` rất lớn): không còn duyệt được từng `x`, cần tính chất bit / DP chữ số.
2. **Number of 1 Bits** (191), **Power of Two** (231): bit cơ bản.
3. **Counting Bits** (338), **Sum of Two Integers** (371): thao tác bit.
4. **Find the Original Array of Prefix Xor** (2433): XOR.
5. Biến thể: thay điều kiện `n & x == 0` bằng `n & x == n` (x chứa mọi bit của `n`) hoặc `n | x == n`.
6. Biến thể: trả về **số lượng** (không phải tổng) số tương thích, hoặc số tương thích **lớn nhất**.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Duyệt khoảng | `x` từ `max(1, n-k)` đến `n+k` | `O(k)` | `O(1)` | **Nộp bài**, đơn giản nhất |
| 2. Duyệt rộng + lọc | `x` từ 1, kiểm cả hai điều kiện | `O(n + k)` | `O(1)` | Tránh nhầm cận |
| 3. Tập con mặt nạ | duyệt tập con của `~n` | `O(2^b)` | `O(1)` | Khi `k` lớn, bit 1 của `n` nhiều |
| 4. Một dòng `sum` | generator | `O(k)` | `O(1)` | Code ngắn |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().sumOfGoodIntegers(1, 2)` và `Solution().sumOfGoodIntegers(4, 3)` ra gì?

**Bài 2 (tính tay):** với `n = 6, k = 4`, liệt kê `x` trong khoảng, nhị phân, giữ/bỏ, tổng.

**Bài 3 (điền TODO):** hoàn thành `count_compatible(n, k)`: **số lượng** số tương thích.

**Bài 4 (tự viết lại):** viết `largest_compatible(n, k)`: số tương thích lớn nhất, hoặc `-1` nếu không có.

**Bài 5 (mini challenge):** viết `sum_compatible_or(n, k)`: đổi điều kiện thành `n & x == n` (`x` chứa **mọi** bit 1 của `n`) rồi trả về tổng các `x` thỏa.

In [22]:
# Bài 1: đoán rồi chạy
print(Solution().sumOfGoodIntegers(1, 2), Solution().sumOfGoodIntegers(4, 3))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def count_compatible_todo(n, k):
    # TODO: đếm x trong [max(1, n - k), n + k] mà n & x == 0
    return 0

2 6


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: `n = 1` (`001`), khoảng `[1, 3]`: bit 0 của `x` phải là 0. `n = 4` (`100`), khoảng `[1, 7]`.

**Mức 2 (vừa):** Bài 3: thay `ans += x` bằng `ans += 1`. Bài 4: duyệt từ `n + k` xuống `max(1, n - k)`, gặp số đầu tiên thỏa thì trả về. Bài 5: đổi điều kiện thành `n & x == n`.

**Mức 3 (gần đáp án):** Bài 1: `n=1,k=2`: `x=2` (`10 & 01 = 0`), `x=3` không, `x=1` không -> `2`; `n=4,k=3`: `x=1,2,3` thỏa, `x=5,6,7` không -> `6`. Bài 2: `n = 6` (`110`), khoảng `[2, 10]`: chỉ `x = 8 (1000)` và `x = 9 (1001)` không chung bit với `0110` -> tổng `17`.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1, 2
assert Solution().sumOfGoodIntegers(1, 2) == 2 and Solution().sumOfGoodIntegers(4, 3) == 6
kept6 = [x for x in range(max(1, 6 - 4), 6 + 4 + 1) if x & 6 == 0]
assert kept6 == [8, 9] and Solution().sumOfGoodIntegers(6, 4) == 17

# Bài 3
def count_compatible(n, k):
    return sum(1 for x in range(max(1, n - k), n + k + 1) if n & x == 0)

assert count_compatible(2, 3) == 3 and count_compatible(5, 1) == 0

# Bài 4
def largest_compatible(n, k):
    for x in range(n + k, max(1, n - k) - 1, -1):
        if n & x == 0:
            return x
    return -1

assert largest_compatible(2, 3) == 5 and largest_compatible(5, 1) == -1

# Bài 5
def sum_compatible_or(n, k):
    return sum(x for x in range(max(1, n - k), n + k + 1) if n & x == n)

assert sum_compatible_or(2, 3) == 2 + 3 and sum_compatible_or(5, 1) == 5
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

### Template

```python
class Solution:
    def sumOfGoodIntegers(self, n: int, k: int) -> int:
        ans = 0
        for x in range(max(1, n - k), n + k + 1):   # x dương, |x - n| <= k, gồm cả hai đầu
            if n & x == 0:                          # hai tập bit rời nhau
                ans += x
        return ans
```

### Checklist

- [ ] Đã đọc kỹ constraints (độ dài, miền giá trị) và biết thuật toán chạy được trong giới hạn đó.
- [ ] Đã chạy tay ít nhất **một ví dụ của đề** và một **trường hợp biên** (nhỏ nhất, lớn nhất, giá trị đặc biệt).
- [ ] Đúng **tên class / tên hàm / kiểu trả về** như skeleton LeetCode (đúng `bool`, `int`, `list`...).
- [ ] Không dùng `input()`, không `print` thừa trong code nộp.
- [ ] Không làm hỏng dữ liệu đầu vào nếu đề không cho phép (sao chép trước khi sửa).
- [ ] Đã so sánh với một cách brute force trên input ngẫu nhiên nhỏ.
- [ ] Cận dưới là `max(1, n - k)` (dương), cận trên `n + k + 1` trong `range`.
- [ ] Điều kiện là `n & x == 0` (AND), đã thử ca `n - k <= 0` (ví dụ `n = 2, k = 6`).